# Colonnes et modalités de Citiz AURA final

Ce notebook décrit **toutes les colonnes** de `data/release/citiz_aura_final.csv`, avec leurs types, leurs modalités observées, leurs effectifs et les données manquantes. Une ligne du fichier représente une station.

Utiliser le noyau **Python (IGN / OSM)** (`ign-osm/.venv`). Redémarrer le noyau puis exécuter toutes les cellules dans l'ordre. Aucun téléchargement ni modification des sources n'est nécessaire.

Une **modalité** est une valeur distincte observée. Pour une variable numérique, les bornes minimum et maximum résument les valeurs : elles ne garantissent pas que chaque valeur intermédiaire existe. Une cellule vide signifie « inconnu ou non calculé », jamais zéro.

In [1]:
from pathlib import Path
import hashlib
import pandas as pd
from IPython.display import display, Markdown

current = Path.cwd().resolve()
candidates = [current, *current.parents]
project_root = next(
    (path for path in candidates if (path / "regional-data/data/release/citiz_aura_final.csv").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Run from the repository or one of its subdirectories; release CSV required.")

source_path = project_root / "regional-data/data/release/citiz_aura_final.csv"
# Preserve original labels and distinguish empty cells from valid text such as NA.
data = pd.read_csv(source_path, encoding="utf-8-sig", dtype="string", keep_default_na=False)
data = data.replace("", pd.NA)
source_hash = hashlib.sha256(source_path.read_bytes()).hexdigest()
display(pd.DataFrame([{
    "source": str(source_path.relative_to(project_root)),
    "rows": len(data), "columns": len(data.columns), "sha256": source_hash,
}]))

,source,rows,columns,sha256
0,regional-data/data/release/citiz_aura_final.csv,612,50,684883fb65ed3c12e20a4923ae917a6e076629666617db...


## 1. Inventaire complet des colonnes

Le type est déduit des valeurs présentes : booléen (`True`/`False`), entier, décimal ou texte. La date d'extraction est identifiée séparément. Une colonne entièrement vide a un type **indéterminé** : le CSV ne suffit pas à connaître son type attendu.

`distinct_count` exclut les cellules vides. `missing_count` et `missing_pct` les comptabilisent séparément. Les colonnes très variées affichent un aperçu ; la section suivante permet de consulter toutes leurs modalités.

In [2]:
def infer_type(series):
    values = series.dropna()
    if values.empty:
        return "undetermined"
    if set(values.unique()).issubset({"True", "False"}):
        return "boolean"
    if series.name == "extracted_at":
        dates = pd.to_datetime(values, errors="coerce", utc=True)
        if dates.notna().all():
            return "datetime"
    numeric = pd.to_numeric(values, errors="coerce")
    if numeric.notna().all():
        return "integer" if numeric.mod(1).eq(0).all() else "decimal"
    return "text"

def summarize_column(name, series):
    kind = infer_type(series)
    values = series.dropna()
    unique = sorted(values.unique().tolist())
    numeric = pd.to_numeric(values) if kind in {"integer", "decimal"} else None
    return {
        "column": name, "observed_type": kind,
        "non_missing_count": int(series.notna().sum()),
        "missing_count": int(series.isna().sum()),
        "missing_pct": round(series.isna().mean() * 100, 2),
        "distinct_count": len(unique),
        "minimum": numeric.min() if numeric is not None else pd.NA,
        "maximum": numeric.max() if numeric is not None else pd.NA,
        "modalities_preview": " | ".join(unique[:12]) + (" | …" if len(unique) > 12 else ""),
    }

column_summary = pd.DataFrame([
    summarize_column(name, data[name]) for name in data.columns
])
with pd.option_context("display.max_rows", None, "display.max_colwidth", 100):
    display(column_summary)

,column,observed_type,non_missing_count,missing_count,missing_pct,distinct_count,minimum,maximum,modalities_preview
0,station_uid,text,612,0,0.00,612,<NA>,<NA>,offre-et-disponibilite-temps-reel-du-service-citiz-en-autopartage-metropole-de-lyon:1115 | offre...
1,station_name,text,612,0,0.00,612,<NA>,<NA>,Parking de l'église | Place Paul Doumer | 10 Rue du Printemps | 20 rue des Tilleuls | Agence ...
2,latitude,decimal,612,0,0.00,606,44.816765,46.372982,44.81676483154297 | 44.826425 | 44.91271209716797 | 44.91986083984375 | 44.92478942871094 | 44.9...
3,longitude,decimal,612,0,0.00,605,3.196774,6.706525,3.196773588757777 | 4.380849838256836 | 4.383253 | 4.384103775024414 | 4.385979175567627 | 4.386...
4,irve_count_500m,undetermined,0,612,100.00,0,<NA>,<NA>,
5,amenities_count_500m,integer,612,0,0.00,142,0,302,0 | 1 | 10 | 101 | 102 | 11 | 110 | 111 | 114 | 115 | 118 | 12 | …
6,amenities_count_500m_straight_line,integer,612,0,0.00,229,0,623,0 | 1 | 10 | 100 | 101 | 103 | 105 | 106 | 109 | 11 | 110 | 111 | …
7,amenities_unreachable_or_unsnapped_500m,integer,612,0,0.00,252,0,660,0 | 1 | 10 | 101 | 102 | 103 | 104 | 105 | 106 | 107 | 109 | 11 | …
8,bike_parking_count_300m,integer,612,0,0.00,81,0,109,0 | 1 | 10 | 100 | 102 | 109 | 11 | 12 | 13 | 14 | 15 | 16 | …
9,bike_parking_count_300m_straight_line,integer,612,0,0.00,133,0,176,0 | 1 | 10 | 103 | 104 | 105 | 107 | 108 | 109 | 11 | 111 | 113 | …


## 2. Modalités exactes et effectifs

Chaque tableau donne une modalité, son effectif et sa part dans les **stations du fichier**, valeurs manquantes comprises au dénominateur. La colonne `is_missing` distingue explicitement une cellule vide d'un texte.

Les tableaux ci-dessous montrent les catégories et les colonnes entièrement vides. Les colonnes numériques sont décrites par leurs bornes dans l'inventaire ; leurs valeurs exactes sont consultables dans la section 3. Les identifiants et noms sont également consultables à cet endroit pour éviter d'afficher deux fois 612 lignes.

In [3]:
def modality_table(column):
    if column not in data.columns:
        raise KeyError(f"Unknown column: {column}")
    counts = data[column].value_counts(dropna=False)
    result = counts.rename_axis("value").reset_index(name="count")
    result["is_missing"] = result["value"].isna()
    result["percentage"] = (result["count"] / len(data) * 100).round(2)
    return result

categorical_columns = column_summary.loc[
    column_summary["observed_type"].isin(["text", "boolean", "datetime", "undetermined"]),
    "column",
].tolist()
for column in categorical_columns:
    if column in {"station_uid", "station_name"}:
        continue
    display(Markdown(f"**`{column}`**"))
    with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
        display(modality_table(column))

**`irve_count_500m`**

,value,count,is_missing,percentage
0,<NA>,612,True,100.0


**`transit_departures_peak_hour`**

,value,count,is_missing,percentage
0,<NA>,612,True,100.0


**`pedestrian_network_distance_m`**

...

,value,count,is_missing,percentage
0,True,612,False,100.0


**`extracted_at`**

,value,count,is_missing,percentage
0,2026-10-19 14:43:54.580579+00:00,612,False,100.0


**`station_reference_match`**

,value,count,is_missing,percentage
0,both,612,False,100.0


## 3. Explorer une colonne au choix

Modifier `selected_column` puis exécuter cette cellule pour obtenir **toutes les valeurs distinctes et leurs effectifs**, y compris pour une variable numérique ou un identifiant. Le tri suit les effectifs décroissants.

In [4]:
selected_column = "capacity"
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(modality_table(selected_column))

,value,count,is_missing,percentage
0,2,333,False,54.41
1,1,112,False,18.3
2,3,102,False,16.67
3,4,33,False,5.39
4,5,14,False,2.29
5,7,5,False,0.82
6,6,4,False,0.65
7,0,3,False,0.49
8,8,2,False,0.33
9,12,1,False,0.16


## 4. Données manquantes et colonnes constantes

Une colonne constante n'a qu'une seule valeur renseignée distincte ; elle peut néanmoins comporter des cellules vides. Une colonne entièrement vide est présentée à part. Ces constats décrivent cet instantané et peuvent changer lors d'une prochaine extraction.

In [5]:
display(Markdown("**Colonnes avec des valeurs manquantes**"))
display(column_summary.loc[column_summary["missing_count"].gt(0),
    ["column", "non_missing_count", "missing_count", "missing_pct"]])
display(Markdown("**Colonnes constantes parmi les valeurs renseignées**"))
display(column_summary.loc[column_summary["distinct_count"].eq(1),
    ["column", "modalities_preview", "missing_count"]])
display(Markdown("**Bilan calculé**"))
display(pd.DataFrame([{
    "stations": len(data), "columns": len(data.columns),
    "fully_empty_columns": int(column_summary["non_missing_count"].eq(0).sum()),
    "columns_with_missing_values": int(column_summary["missing_count"].gt(0).sum()),
    "duplicate_station_ids": int(data["station_uid"].duplicated().sum()),
}]))

**Colonnes avec des valeurs manquantes**

,column,non_missing_count,missing_count,missing_pct
4,irve_count_500m,0,612,100.00
11,transit_departures_peak_hour,0,612,100.00
12,pedestrian_network_distance_m,0,612,100.00
16,accidents_count_500m,0,612,100.00
25,public_parking_spaces_300m,57,555,90.69
41,city,0,612,100.00


**Colonnes constantes parmi les valeurs renseignées**

,column,modalities_preview,missing_count
24,walking_network_status,matched,0
31,irve_count_500m_status,charging_units_not_defined_osm_sites_only,0
32,transit_departures_peak_hour_status,missing_gtfs_service_date_and_peak_window,0
33,pedestrian_network_distance_m_status,missing_origins_destinations_and_regional_graph,0
34,accidents_count_500m_status,missing_accident_data_and_period,0
35,osm_count_status,observed_mapped_objects_not_exhaustive,0
36,osm_extraction_has_geometry_issues,True,0
43,region,Auvergne-Rhône-Alpes,0
47,is_renting,True,0
48,extracted_at,2026-10-19 14:43:54.580579+00:00,0


**Bilan calculé**

,stations,columns,fully_empty_columns,columns_with_missing_values,duplicate_station_ids
0,612,50,5,6,0


## 5. Lire les résultats correctement

- Les comptages OSM décrivent des **objets recensés**, sans garantir l'exhaustivité sur le terrain.
- Les indicateurs principaux utilisent le réseau piéton ; le suffixe `_straight_line` indique une comparaison à vol d'oiseau. Les rayons sont indiqués dans les noms (`300m`, `500m`, `1km`).
- Les objets de stationnement vélo et les sites de recharge ne représentent pas nécessairement des places vélo, des bornes ou des points de recharge.
- `public_parking_spaces_known_300m` est une somme partielle des capacités connues. `public_parking_spaces_300m` reste vide lorsque les accès ou capacités ne permettent pas de conclure.
- Les statuts expliquent les indicateurs non calculés et les limites de couverture.

Consulter le [dictionnaire de la release](../data/release/DATA_DICTIONARY.md) et les [sources](../data/release/SOURCES.md) pour les règles détaillées. Les résultats affichés dans ce notebook proviennent du CSV local, dont l'empreinte SHA-256 figure en tête.